In [1]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [2]:
from langchain_ollama.chat_models import ChatOllama
from langchain.agents import create_agent

model = ChatOllama(model="qwen3.5")

agent = create_agent(
    model=model,
    context_schema=ColourContext
)

d:\workspace\AI\langchain-academy\intro-2-langchain\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [3]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

In [4]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='98e0c79c-c83c-4026-80c9-45f6b20fe627'),
              AIMessage(content="I don't know! I don't have access to your personal memories or preferences unless you tell me.\n\nBut I'd be happy to learn it! If you want to share, I'll remember your favorite color for our conversation. Do you want to tell me?", additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-03T07:00:32.9988797Z', 'done': True, 'done_reason': 'stop', 'total_duration': 20055998400, 'load_duration': 115546900, 'prompt_eval_count': 16, 'prompt_eval_duration': 258009200, 'eval_count': 1962, 'eval_duration': 18790402100, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019d5224-700e-75c0-a5de-558d82526f26-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 1962, 'total_tokens': 1978})]}


### Accessing Context

In [5]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [6]:
agent = create_agent(
    model,
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [10]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

d:\workspace\AI\langchain-academy\intro-2-langchain\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='699bf8ab-921c-48a5-8059-e9eed93dd866'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-03T06:51:14.8282029Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1006195400, 'load_duration': 122555800, 'prompt_eval_count': 311, 'prompt_eval_duration': 259527700, 'eval_count': 58, 'eval_duration': 566383900, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019d521c-361d-7c63-af44-76414772f3d9-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': '095a8b17-6fa0-403a-a026-a4453796f029', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 311, 'output_tokens': 58, 'total_tokens': 369}),
              ToolMessage(content='blue', name='get_favourite_colour', id='30903a72-c92f-4a79-abdd-d6d19ef9a6d1', tool_call_id='095a8b17-6fa0-403

In [7]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What colour do i not like?")]},
    context=ColourContext(favourite_colour="green", least_favourite_colour="red")
)

pprint(response)

d:\workspace\AI\langchain-academy\intro-2-langchain\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c..._favourite_colour='red'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What colour do i not like?', additional_kwargs={}, response_metadata={}, id='5d9b472a-255a-4b62-bfd6-5659436a0c0b'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-03T07:00:48.0788756Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1183031000, 'load_duration': 114401300, 'prompt_eval_count': 312, 'prompt_eval_duration': 345598800, 'eval_count': 73, 'eval_duration': 693507800, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019d5224-f4ae-76f0-abc5-ff6f953cdbae-0', tool_calls=[{'name': 'get_least_favourite_colour', 'args': {}, 'id': '06ea63d9-768d-4e27-bb81-75f19a265ba2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 312, 'output_tokens': 73, 'total_tokens': 385}),
              ToolMessage(content='red', name='get_least_favourite_colour', id='963006e1-5af7-49a7-b6dd-90331ec33b25', tool_call_id='06ea63d9